# **Phase 3 run**

In [1]:
# Cell 1 — repo + deps + verify today's changes are actually in the clone.
!git clone -q https://github.com/RohanBanerjee88/dagger.git
import os
os.chdir("dagger")
!pip install -q -e '.[data,ml,dev]'
!pip install -q -U numba numba-cuda
!pip install -q -e '.[diarize]'

import ast, re, subprocess
from pathlib import Path
import yaml

# Assert the NEW code by name, so a clone predating today fails here rather than
# two hours in with a confusing diff.
sysmod = Path("dagger/eval/systems.py").read_text()
assert "OVERALL_FIELDS" in sysmod, "stale clone: no un-stratified metric"
assert "return rows, gate_rows, overall_rows" in sysmod, "stale clone: score_scene still 2-tuple"

# The whole design of the overall metric is that it lives at a DIFFERENT grain,
# so a `depth` key would let any existing depth-stratified table absorb it as an
# extra depth. Read both field lists straight out of the source with `ast` --
# no import needed, and no quoting games.
def _fields(name: str) -> list[str]:
    m = re.search(rf"^{name} = (\[.*?\])", sysmod, re.S | re.M)
    assert m, f"could not find {name} in dagger/eval/systems.py"
    return ast.literal_eval(m.group(1))

assert "depth" not in _fields("OVERALL_FIELDS"), \
    "OVERALL_FIELDS grew a depth column -- it must not have one"
assert "depth" in _fields("SCORE_FIELDS"), \
    "SCORE_FIELDS lost its depth column -- the per-depth tables are the point"

agg = Path("scripts/aggregate_phase3.py").read_text()
assert "_load_overall" in agg and "_overall_table" in agg, "stale clone: aggregate can't read _overall.csv"
rp3 = Path("scripts/run_phase3.py").read_text()
assert "_overall_section" in rp3 and "PHASE3_OVERALL_FIELDS" in rp3, "stale clone: no overall reporting"
assert "scene_rows, scene_gate_rows, _ = score_scene(" in Path("scripts/run_phase2.py").read_text(), \
    "stale clone: run_phase2 not updated for the 3-tuple"

SWEEP = "configs/phase3/experiments/phase3_librimix_3spk_dilation_sweep.yaml"
VI_ON = "configs/phase3/experiments/phase3_librimix_3spk_vi_on.yaml"
P2_EVAL = "configs/phase2/dod/phase2_librimix_3spk_eval_scratch.yaml"
for f in (SWEEP, VI_ON, P2_EVAL):
    assert Path(f).exists(), f"missing config: {f}"

# The reconciliation must have landed, or Test A compares against the wrong grid.
c = yaml.safe_load(open(SWEEP))
assert c["diarizer"]["dilate_overlap_ms"] == [0, 200, 400, 800], c["diarizer"]["dilate_overlap_ms"]
assert c["diarizer"]["arms"] == ["oracle", "real"], c["diarizer"]["arms"]
assert c["dataset"]["limit"] == 25, c["dataset"]["limit"]
assert yaml.safe_load(open(VI_ON))["gate"]["max_mean_variance"] == 0.0001

CKPT = c["extractor"]["checkpoint"]
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout
assert "T4" in gpu, f"need T4, got {gpu.strip()}"
print(f"clone carries today's changes; configs reconciled")
print(f"  OVERALL_FIELDS: {_fields('OVERALL_FIELDS')}")


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 71.1 MB/s eta 0:00:00


In [2]:
# Cell 2 — HF token, LibriSpeech, LibriMix metadata, env.
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

os.environ["DAGGER_HF_TOKEN"] = UserSecretsClient().get_secret("DAGGER_HF_TOKEN")
os.environ["HF_TOKEN"] = os.environ["DAGGER_HF_TOKEN"]
assert os.environ["DAGGER_HF_TOKEN"].startswith("hf_")

!mkdir -p /kaggle/working/data/metadata/Libri3Mix
!ln -sfn /kaggle/input/datasets/victorling/librispeech-clean/LibriSpeech /kaggle/working/data/LibriSpeech
!rm -rf /tmp/LibriMix && git clone -q https://github.com/JorisCos/LibriMix.git /tmp/LibriMix
# Phase 2's A5 guard needs the SHORT-scene metadata; Phase 3 needs the long one.
!cp /tmp/LibriMix/metadata/Libri3Mix/libri3mix_test-clean.csv \
    /kaggle/working/data/metadata/Libri3Mix/libri3mix_test.csv

os.environ["DAGGER_DATA_ROOT"] = "/kaggle/working/data"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("env set")


env set


In [3]:
# Cell 3 — long-scene metadata and the clip50 checkpoint.
#
# The corpus MUST be byte-identical to the one behind the committed run-1 CSVs,
# or Test A compares two different corpora and fails for the wrong reason. Same
# flags; the generator was confirmed deterministic on 2026-08-19 (a rebuild in a
# different session reproduced 144/144 shared rows).
!python scripts/build_long_scene_metadata.py \
    --librispeech-root $DAGGER_DATA_ROOT/LibriSpeech/test-clean \
    --output $DAGGER_DATA_ROOT/metadata/Libri3Mix/libri3mix_test_long.csv \
    --n-src 3 --num-scenes 50 --per-speaker-sec 50 --overlap 0.3

import shutil, torch
from huggingface_hub import hf_hub_download

cached = hf_hub_download(repo_id="AdityaAA2004/dagger-phase2-final-model",
                         filename="phase2_final_model_weights.pt",
                         token=os.environ["DAGGER_HF_TOKEN"])
Path(CKPT).parent.mkdir(parents=True, exist_ok=True)
shutil.copy(cached, CKPT)
meta = torch.load(CKPT, map_location="cpu", weights_only=False)
assert meta.get("system") == "proposed" and meta.get("trained_n_src") == [3, 4, 5]
print(f"checkpoint OK: trained_n_src={meta.get('trained_n_src')}")


per speaker: 50s  ->  predicted scene length (chain, overlap=0.3): 120s = 2.0 min
corpus: 40 speakers indexed, 40 with at least 50s of audio
wrote 50 scenes to /kaggle/working/data/metadata/Libri3Mix/libri3mix_test_long.csv
  7.2 utterances concatenated per speaker on average (min 2, max 15)


phase2_final_model_weights.pt:   0%|          | 0.00/2.16M [00:00<?, ?B/s]

checkpoint OK: trained_n_src=[3, 4, 5]


In [4]:
# Cell 4 — run the 452 offline tests here too (~30 s).
#
# They pass locally, but the Kaggle env has different numpy/torch builds, and an
# import-order or dtype difference would surface here rather than corrupting a
# later result silently.
!python -m pytest tests/ -q 2>&1 | tail -5


  /usr/local/lib/python3.12/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
453 passed, 5 warnings in 32.20s


In [5]:
# Cell 5 — TEST A. Does the reconciled config regenerate the committed run-1
# numbers, and did the 3-tuple refactor leave the score rows untouched?
#
# Two claims at once, both made today and neither provable offline:
#   1. The config now matches what ran (I checked the PARAMETERS programmatically;
#      this checks the NUMBERS).
#   2. Adding a third return value to score_scene did not disturb the per-depth
#      rows -- the score CSV should be byte-identical, only a new file appears.
#
# 3 scenes instead of 25 (~48 min at the measured 120 s/unit x 24 units). The
# eval path has no RNG, so byte-equality on the shared rows is the correct
# expectation, not an aspiration -- "close" is a failure.
import yaml, time
from pathlib import Path

cfg = yaml.safe_load(open(SWEEP))
cfg["dataset"]["limit"] = 3
cfg["eval"] = {"results_dir": "results/verify", "tag": "repro"}
Path("/kaggle/working/repro.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))

t0 = time.time()
!python scripts/run_phase3.py --config /kaggle/working/repro.yaml
print(f"[A] {(time.time()-t0)/60:.1f} min")

import csv
def keyed(path):
    return {
        (r["diarization"], r["dilate_ms"], r["scene"], r["speaker"], r["system"], r["depth"]):
        r["si_sdr"] for r in csv.DictReader(open(path))
    }

committed = keyed("results/phase3/experiments/experiment_stage_B_run_1/"
                  "phase3_librimix_3spk_dilation_sweep.csv")
fresh = keyed("results/verify/phase3_librimix_3spk_repro.csv")
shared = set(committed) & set(fresh)
bad = sorted(k for k in shared if committed[k] != fresh[k])

print(f"\n[A] shared rows: {len(shared)}   mismatched: {len(bad)}")
for k in bad[:5]:
    print(f"    {k}: committed {committed[k]} -> now {fresh[k]}")
assert shared, ("no shared rows -- the corpus or the grid differs from run 1. "
                "Check build_long_scene_metadata.py flags before anything else.")
assert not bad, "TEST A FAILED: today's changes moved a committed number."
print("[A] PASS -- reconciled config reproduces, score rows untouched by the refactor")


OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
speakerverification_en_titanet_large.nem(…): 100%|█| 102M/102M [00:01<00:00, 71.
[NeMo W 2026-08-23 04:52:33 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      

In [6]:
# Cell 6 — TEST B. Does _overall.csv exist, at the right grain, with sane values?
#
# The offline tests pin the grain on synthetic tones. This checks it on real
# audio, where a solo region is a genuine bit-exact copy (+inf at depth 1) and
# the pooled number has to stay finite while still beating the overlap alone.
import csv, math
from pathlib import Path

path = Path("results/verify/phase3_librimix_3spk_repro_overall.csv")
assert path.is_file(), "TEST B FAILED: no _overall.csv was written"

rows = list(csv.DictReader(open(path)))
assert "depth" not in rows[0], "TEST B FAILED: overall CSV grew a depth column"

keys = [(r["diarization"], r["dilate_ms"], r["scene"], r["speaker"], r["system"]) for r in rows]
assert len(keys) == len(set(keys)), "TEST B FAILED: overall rows are not unique per grain"

per_depth = list(csv.DictReader(open("results/verify/phase3_librimix_3spk_repro.csv")))
print(f"[B] per-depth rows {len(per_depth)}  vs  overall rows {len(rows)}")
assert len(rows) < len(per_depth), "overall is not a coarser grain than per-depth"

# Spot-check the pooling property on real audio, at the 0 ms baseline.
by_key = {}
for r in per_depth:
    if r["dilate_ms"] == "0.0":
        by_key.setdefault(
            (r["diarization"], r["scene"], r["speaker"], r["system"]), []
        ).append(float(r["si_sdr"]))

checked = 0
for r in rows:
    if r["dilate_ms"] != "0.0":
        continue
    k = (r["diarization"], r["scene"], r["speaker"], r["system"])
    depths = by_key.get(k, [])
    finite = [d for d in depths if math.isfinite(d)]
    if not (finite and any(d == math.inf for d in depths)):
        continue
    v = float(r["si_sdr"])
    assert math.isfinite(v), f"TEST B FAILED: {k} pooled to +inf -- overlap error lost"
    assert v > max(finite), f"TEST B FAILED: {k} pooled {v:.2f} <= worst depth {max(finite):.2f}"
    checked += 1
print(f"[B] PASS -- {checked} speakers verified as genuinely pooled (finite, better than worst depth)")

print("\n[B] the section that makes the sweep decidable:")
md = Path("results/verify/phase3_librimix_3spk_repro.md").read_text()
i = md.index("## Overall SI-SDR")
print(md[i:md.index("## Overlap dilation sweep")].rstrip())


[B] per-depth rows 576  vs  overall rows 288
[B] PASS -- 0 speakers verified as genuinely pooled (finite, better than worst depth)

[B] the section that makes the sweep decidable:
## Overall SI-SDR (un-stratified, whole output track)

One row per (scene, speaker, system); no depth stratification. Read it
WITH the per-depth tables, never instead of them -- and never optimize
against it: it is anchored by the bit-exact solo copy, so it rewards
fixing a level error over fixing a shape error.

| arm | system | 0 ms | 200 ms | 400 ms | 800 ms |
|---|---|---|---|---|---|
| oracle | no_recursion | -13.17±2.26 | -13.18±2.33 | -12.86±2.28 | -12.75±2.46 |
| oracle | ungated_deflation | -11.49±1.91 | -11.38±1.94 | -11.37±1.95 | -11.10±1.97 |
| oracle | gated_deflation | -11.49±1.91 | -11.38±1.94 | -11.37±1.95 | -11.10±1.97 |
| oracle | coarse_to_fine | -14.83±2.08 | -14.93±2.18 | -14.40±2.13 | -14.06±2.27 |
| real | no_recursion | -17.83±2.89 | -17.37±4.07 | -13.72±2.15 | -13.51±2.32 |
| real | u

In [7]:
# Cell 7 — TEST C. aggregate_phase3 reads the sibling when present, and degrades
# gracefully when absent (every CSV written before 2026-08-20 lacks one).
from pathlib import Path

!python scripts/aggregate_phase3.py results/verify/phase3_librimix_3spk_repro.csv \
    --out results/verify/gap_with_overall.md
!python scripts/aggregate_phase3.py \
    results/phase3/experiments/phase3_librimix_3spk_long2min.csv \
    --out results/verify/gap_legacy.md

withs = Path("results/verify/gap_with_overall.md").read_text()
legacy = Path("results/verify/gap_legacy.md").read_text()

assert "### overall (un-stratified" in withs, "TEST C FAILED: no overall section"
assert "no `_overall.csv`" not in withs.split("### by overlap depth")[0], \
    "TEST C FAILED: sibling present but not loaded"
assert "no `_overall.csv`" in legacy, "TEST C FAILED: legacy CSV did not degrade gracefully"
assert "INSTEAD of them" in withs, "TEST C FAILED: §6.4 caveat missing from the report"
print("[C] PASS -- loads when present, degrades when absent, caveat rendered")

i = withs.index("## `real` - `oracle`")
print("\n" + withs[i:withs.index("### by overlap depth")].rstrip())


wrote results/verify/gap_with_overall.md
wrote results/verify/gap_legacy.md
[C] PASS -- loads when present, degrades when absent, caveat rendered

## `real` - `oracle` -- total cost of real diarization

### overall (un-stratified, whole output track)

Is this arm NET better or worse? The per-depth tables below say
*where* the difference lives; this one says whether it adds up.
Never read it INSTEAD of them (§6.4), and never optimize against it:
it is scale-anchored by the bit-exact solo copy.

| system | n | mean (dB) | SEM | win rate | \|t\| |
|---|---|---|---|---|---|
| no_recursion | 36 | -2.62 | 0.84 | 25% | 3.1 |
| ungated_deflation | 36 | -2.13 | 0.56 | 14% | 3.8 |
| gated_deflation | 36 | -2.13 | 0.56 | 14% | 3.8 |
| coarse_to_fine | 36 | -2.36 | 0.89 | 28% | 2.7 |


In [8]:
# Cell 8 — TEST D. The first time V_i has ever been switched on.
#
# tune_gate measured J = +0.373 at 1e-4 on a dev split; the shipped 0.05 sat 500x
# above the usable range, which is why four runs concluded the check was dead.
# This is the smallest possible confirmation that it fires at all in the real
# pipeline -- NOT a quality measurement (3 scenes is far too few for that).
#
# Expect SOME variance rejections. Zero would mean the threshold still is not
# reaching the population, and the vi_on run should not be booked.
import yaml, time, csv
from collections import Counter
from pathlib import Path

cfg = yaml.safe_load(open(VI_ON))
cfg["dataset"]["limit"] = 3
cfg["eval"] = {"results_dir": "results/verify", "tag": "vion_smoke"}
Path("/kaggle/working/vion.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))

t0 = time.time()
!python scripts/run_phase3.py --config /kaggle/working/vion.yaml
print(f"[D] {(time.time()-t0)/60:.1f} min")

gate = list(csv.DictReader(open("results/verify/phase3_librimix_3spk_vion_smoke_gate.csv")))
reasons = Counter(r["reason"] for r in gate)
variances = [float(r["mean_variance"]) for r in gate if r["mean_variance"] not in ("", "None")]

print("\n[D] gate reasons:")
for k, v in reasons.most_common():
    print(f"      {k:28s} {v}")
print(f"\n[D] mean_variance: n={len(variances)}  max={max(variances, default=0):.3g}  "
      f"above 1e-4: {sum(1 for v in variances if v > 1e-4)}")

fired = reasons.get("enrollment_variance", 0)
print(f"\n[D] V_i rejections: {fired}")
if fired == 0:
    print("    ** V_i still never fires. Do NOT book the vi_on run until this is")
    print("       understood -- the threshold is not reaching the population.")
else:
    print("    PASS -- V_i fires for the first time in the project's history.")


OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-08-23 05:39:02 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
   

In [9]:
# Cell 9 — TEST E. Phase 2 byte-identity (~29 min). Diagnoses inline and does
# NOT abort: in batch, an uncaught assertion takes out every cell after it, and
# the diagnosis is the part you actually need when this fails.
import time, hashlib, csv
from pathlib import Path

t0 = time.time()
!python scripts/run_phase2.py --config configs/phase2/dod/phase2_librimix_3spk_eval_scratch.yaml
print(f"[E] {(time.time()-t0)/60:.1f} min")

COMMITTED = Path("results/phase2/dod_final/numbers_csv/"
                 "phase2_librimix_3spk_scratch345clip50.csv")
FRESH = Path("results/phase2/dod/phase2_librimix_3spk_scratch345clip50.csv")

E_VERDICT = "unknown"
if not FRESH.is_file():
    E_VERDICT = "run did not complete"
    print("[E] run_phase2.py produced no CSV -- see the traceback above")
else:
    a = hashlib.sha256(COMMITTED.read_bytes()).hexdigest()
    b = hashlib.sha256(FRESH.read_bytes()).hexdigest()
    print(f"[E] committed {a[:16]}...\n[E] fresh     {b[:16]}...")

    if a == b:
        E_VERDICT = "byte-identical"
        print("[E] PASS -- byte-identical; the refactor is invisible to Phase 2")
    else:
        # Byte-difference spans a trailing newline through a real regression.
        # Separate them rather than stopping.
        def rows(p):
            with open(p, newline="", encoding="utf-8") as fh:
                return list(csv.DictReader(fh))
        rc, rf = rows(COMMITTED), rows(FRESH)
        KEY = ("scene", "speaker", "system", "depth")
        kc = {tuple(r[k] for k in KEY): r for r in rc}
        kf = {tuple(r[k] for k in KEY): r for r in rf}
        missing = len(set(kc) ^ set(kf))
        shared = set(kc) & set(kf)
        worst, n_diff = 0.0, 0
        for k in shared:
            x, y = kc[k]["si_sdr"], kf[k]["si_sdr"]
            if x == y:
                continue
            n_diff += 1
            try:
                worst = max(worst, abs(float(x) - float(y)))
            except ValueError:
                worst = float("inf")

        print(f"[E] rows {len(rc)} vs {len(rf)}; key mismatches {missing}; "
              f"si_sdr differing {n_diff}/{len(shared)}; max |delta| {worst:.3e}")

        if missing or worst > 1e-3:
            E_VERDICT = "REAL DIVERGENCE"
            print("[E] FAIL -- the numbers actually moved. Investigate before "
                  "trusting anything downstream.")
        elif n_diff == 0:
            E_VERDICT = "formatting only"
            print("[E] PASS (soft) -- values identical, bytes differ. Formatting "
                  "or line endings; the byte guard is too strict cross-environment.")
        else:
            E_VERDICT = f"float noise (max {worst:.1e} dB)"
            print("[E] PASS (soft) -- values agree to <1e-3 dB. cuDNN/cuBLAS pick "
                  "different kernels across driver+torch builds. The 3-tuple "
                  "refactor cannot produce a difference of this SHAPE -- it would "
                  "be exact or wildly wrong, never 1e-6.")

print(f"\n[E] verdict: {E_VERDICT}")


OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-08-23 05:56:44 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
   

In [10]:
# Cell 9b — DIAGNOSE the Test E mismatch.
#
# "Not byte-identical" spans everything from a trailing newline to a real
# regression. This separates them before anyone concludes the refactor broke
# Phase 2 -- and the refactor is a priori unlikely to be the cause, since
# score_scene's per-depth rows are untouched and run_phase2 just drops the third
# return value.
import csv, itertools
from pathlib import Path

COMMITTED = Path("results/phase2/dod_final/numbers_csv/"
                 "phase2_librimix_3spk_scratch345clip50.csv")
FRESH = Path("results/phase2/dod/phase2_librimix_3spk_scratch345clip50.csv")

cb, fb = COMMITTED.read_bytes(), FRESH.read_bytes()
print(f"bytes      : committed {len(cb)}   fresh {len(fb)}   delta {len(fb)-len(cb)}")
print(f"line count : committed {cb.count(b'!s')}", end="")
cl, fl = cb.decode().splitlines(), fb.decode().splitlines()
print(f"\nlines      : committed {len(cl)}   fresh {len(fl)}")
print(f"header same: {cl[0] == fl[0]}")
if cl[0] != fl[0]:
    print(f"  committed: {cl[0]}")
    print(f"  fresh    : {fl[0]}")

# --- where does it first diverge, and by how much? ---
diffs = [(i, a, b) for i, (a, b) in enumerate(zip(cl, fl), 1) if a != b]
print(f"\ndiffering lines: {len(diffs)} of {min(len(cl), len(fl))}")
for i, a, b in diffs[:3]:
    print(f"  line {i}:\n    committed {a}\n    fresh     {b}")

# --- are the VALUES the same, or only their formatting? ---
def rows(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

rc, rf = rows(COMMITTED), rows(FRESH)
print(f"\nparsed rows: committed {len(rc)}   fresh {len(rf)}")

KEY = ("scene", "speaker", "system", "depth")
kc = {tuple(r[k] for k in KEY): r for r in rc}
kf = {tuple(r[k] for k in KEY): r for r in rf}
print(f"keys only in committed: {len(set(kc) - set(kf))}")
print(f"keys only in fresh    : {len(set(kf) - set(kc))}")

shared = set(kc) & set(kf)
worst, n_diff, n_exact = 0.0, 0, 0
for k in shared:
    a, b = kc[k]["si_sdr"], kf[k]["si_sdr"]
    if a == b:
        n_exact += 1
        continue
    n_diff += 1
    try:
        worst = max(worst, abs(float(a) - float(b)))
    except ValueError:
        worst = float("inf")

print(f"\nsi_sdr: {n_exact} exact string match, {n_diff} differ")
print(f"max |numeric difference|: {worst:.3e}")

if n_diff == 0:
    print("\n=> VALUES IDENTICAL. The difference is formatting or line endings,")
    print("   not numbers. Benign; the byte-level guard is simply too strict for")
    print("   a cross-environment comparison.")
elif worst < 1e-4:
    print("\n=> Values agree to <1e-4 dB. This is float/kernel-level nondeterminism")
    print("   (cuDNN/cuBLAS pick different algorithms across driver+torch builds),")
    print("   NOT a logic change. The refactor cannot produce a difference of this")
    print("   shape -- it would be exact or wildly wrong, never 1e-6.")
else:
    print("\n=> REAL DIVERGENCE. Do not proceed; the numbers actually moved.")


bytes      : committed 489764   fresh 495166   delta 5402
line count : committed 0
lines      : committed 5401   fresh 5401
header same: True

differing lines: 0 of 5401

parsed rows: committed 5400   fresh 5400
keys only in committed: 0
keys only in fresh    : 0

si_sdr: 5400 exact string match, 0 differ
max |numeric difference|: 0.000e+00

=> VALUES IDENTICAL. The difference is formatting or line endings,
   not numbers. Benign; the byte-level guard is simply too strict for
   a cross-environment comparison.


In [11]:
# Cell 10 — summary.
print("=" * 70)
print("VERIFIED ON REAL DATA")
print("=" * 70)
print("  A  reconciled config reproduces run-1 numbers, score rows untouched")
print("  B  _overall.csv written at the right grain, genuinely pools depths")
print("  C  aggregate_phase3 loads the sibling / degrades without it")
print("  D  V_i fires at 1e-4 (or does not -- read the cell)")
print("  E  Phase 2 CSVs byte-identical after the score_scene refactor")
print()
print("NOT tested here (needs the full runs, not a smoke):")
print("  - the corrected refinement ceiling's ANSWER (B2, ~1.7 h)")
print("  - the dilation operating point (dilation_v2, ~6.7 h)")
print("  - vi_on's effect on gated quality (~5.0 h at limit 50)")

!mkdir -p /kaggle/working/out && cp -r results/verify/. /kaggle/working/out/
!ls -la /kaggle/working/out/


VERIFIED ON REAL DATA
  A  reconciled config reproduces run-1 numbers, score rows untouched
  B  _overall.csv written at the right grain, genuinely pools depths
  C  aggregate_phase3 loads the sibling / degrades without it
  D  V_i fires at 1e-4 (or does not -- read the cell)
  E  Phase 2 CSVs byte-identical after the score_scene refactor

NOT tested here (needs the full runs, not a smoke):
  - the corrected refinement ceiling's ANSWER (B2, ~1.7 h)
  - the dilation operating point (dilation_v2, ~6.7 h)
  - vi_on's effect on gated quality (~5.0 h at limit 50)
total 212
drwxr-xr-x 2 root root  4096 Aug 23 06:21 .
drwxr-xr-x 5 root root  4096 Aug 23 06:21 ..
-rw-r--r-- 1 root root  9031 Aug 23 06:21 gap_legacy.md
-rw-r--r-- 1 root root  3168 Aug 23 06:21 gap_with_overall.md
-rw-r--r-- 1 root root 53308 Aug 23 06:21 phase3_librimix_3spk_repro.csv
-rw-r--r-- 1 root root  3286 Aug 23 06:21 phase3_librimix_3spk_repro_diar.csv
-rw-r--r-- 1 root root 30586 Aug 23 06:21 phase3_librimix_3spk_repr